In [1]:
# STEP 1: MODEL TRAINING AND PERSISTENCE
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from datetime import datetime, timedelta

In [4]:
# 1. LOAD AND PREPARE DATA (from your notebook)
# ============================================================================

print("\n1. Loading dataset...")

# Load the dataset
df = pd.read_csv("C://Users//Raj03//OneDrive//Desktop//Rajesh_UC_Berkley//kraftwerk//Saravwerk//KaggleV2-May-2016.csv")

print(f"Dataset loaded: {len(df):,} records")

# Data preparation 
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df['ScheduledDate'] = df['ScheduledDay'].dt.date
df['AppointmentDate'] = df['AppointmentDay'].dt.date

# Calculate expected shows
df['ExpectedShow'] = 1 - df['No-show'].map({'Yes': 1, 'No': 0})

# Create daily aggregates
daily_volumes = df.groupby('AppointmentDate').agg({
    'ExpectedShow': 'sum',
    'PatientId': 'count'
}).rename(columns={'PatientId': 'TotalAppointments'})

daily_volumes.index = pd.to_datetime(daily_volumes.index)
daily_volumes = daily_volumes.sort_index()

print(f"Daily aggregates created: {len(daily_volumes)} days")


1. Loading dataset...
Dataset loaded: 110,527 records
Daily aggregates created: 27 days


In [5]:
# 2. FEATURE ENGINEERING 
# ============================================================================

print("\n2. Engineering features...")

# Temporal features
daily_volumes['DayOfWeek'] = daily_volumes.index.dayofweek
daily_volumes['DayOfMonth'] = daily_volumes.index.day
daily_volumes['WeekOfYear'] = daily_volumes.index.isocalendar().week
daily_volumes['Month'] = daily_volumes.index.month
daily_volumes['IsWeekend'] = (daily_volumes['DayOfWeek'] >= 5).astype(int)

# Lag features
for lag in [1, 2, 3, 7, 14]:
    daily_volumes[f'Lag_{lag}'] = daily_volumes['ExpectedShow'].shift(lag)

# Rolling statistics
for window in [3, 7, 14]:
    daily_volumes[f'Rolling_Mean_{window}'] = daily_volumes['ExpectedShow'].rolling(window=window).mean()
    daily_volumes[f'Rolling_Std_{window}'] = daily_volumes['ExpectedShow'].rolling(window=window).std()

# Day-of-week encoding
for day in range(7):
    daily_volumes[f'Day_{day}'] = (daily_volumes['DayOfWeek'] == day).astype(int)

# Drop rows with NaN from lag/rolling features
daily_volumes = daily_volumes.dropna()

print(f"Features engineered: {daily_volumes.shape[1]} total features")


2. Engineering features...
Features engineered: 25 total features


In [6]:
# 3. PREPARE TRAINING DATA
# ============================================================================

print("\n3. Preparing training data...")

# Feature list (same as notebook)
feature_columns = [col for col in daily_volumes.columns if col != 'ExpectedShow' and col != 'TotalAppointments']

X = daily_volumes[feature_columns]
y = daily_volumes['ExpectedShow']

print(f"Features: {len(feature_columns)}")
print(f"Training samples: {len(X)}")


3. Preparing training data...
Features: 23
Training samples: 13


In [7]:
# ============================================================================
# 4. TRAIN FINAL MODEL (Ridge with alpha=1)
# ============================================================================

print("\n4. Training Ridge Regression model (α=1)...")

# Create pipeline (same as notebook)
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1, random_state=42))
])

# Train on ALL available data (for deployment)
ridge_pipeline.fit(X, y)

print(" Model training complete")


4. Training Ridge Regression model (α=1)...
✓ Model training complete


In [8]:
# 5. SAVE MODEL AND FEATURE INFO
# ============================================================================

print("\n5. Saving model and metadata...")

# Save the trained pipeline
joblib.dump(ridge_pipeline, 'hospital_staffing_model.pkl')
print(" Model saved: hospital_staffing_model.pkl")

# Save feature column names (critical for prediction)
joblib.dump(feature_columns, 'feature_columns.pkl')
print(" Feature columns saved: feature_columns.pkl")

# Save metadata
metadata = {
    'model_type': 'Ridge Regression',
    'alpha': 1,
    'n_features': len(feature_columns),
    'training_samples': len(X),
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'expected_mape': 5.35,
    'feature_columns': feature_columns
}

joblib.dump(metadata, 'model_metadata.pkl')
print(" Metadata saved: model_metadata.pkl")



5. Saving model and metadata...
✓ Model saved: hospital_staffing_model.pkl
✓ Feature columns saved: feature_columns.pkl
✓ Metadata saved: model_metadata.pkl


In [9]:
# 6. TEST THE SAVED MODEL
# ============================================================================

print("\n6. Testing saved model...")

# Load the model
loaded_model = joblib.load('hospital_staffing_model.pkl')
loaded_features = joblib.load('feature_columns.pkl')
loaded_metadata = joblib.load('model_metadata.pkl')

# Test prediction on last sample
test_sample = X.iloc[-1:].copy()
prediction = loaded_model.predict(test_sample)

print(f" Test prediction successful: {prediction[0]:.1f} patients")


6. Testing saved model...
✓ Test prediction successful: 3709.5 patients


In [10]:
# 7. CREATE SAMPLE DATA FOR STREAMLIT DEMO
# ============================================================================

print("\n7. Creating sample data for demo...")

# Save last 30 days of actual data for comparison
recent_data = daily_volumes.tail(30)[['ExpectedShow']].copy()
recent_data.to_csv('recent_actual_volumes.csv')
print(" Sample data saved: recent_actual_volumes.csv")


7. Creating sample data for demo...
✓ Sample data saved: recent_actual_volumes.csv


In [ ]:
FAST API SETUP

In [12]:
pip install fastapi uvicorn[standard] streamlit plotly pandas scikit-learn joblib pydantic requests


   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
  Attempting uninstall: packaging
   ---------------------------------------- 0/9 [websockets]
   -------- ------------------------------- 2/9 [packaging]
    Found existing installation: packaging 25.0
   -------- ------------------------------- 2/9 [packaging]
    Uninstalling packaging-25.0:
   -------- ------------------------------- 2/9 [packaging]
      Successfully uninstalled packaging-25.0
   -------- ------------------------------- 2/9 [packaging]
   -------- ------------------------------- 2/9 [packaging]
   -------- ------------------------------- 2/9 [p

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 6.0.7 requires ipython!=8.17.1,<9.0.0,>=8.13.0; python_version > "3.8", but you have ipython 9.8.0 which is incompatible.
spyder-kernels 3.0.5 requires ipykernel<7,>=6.29.3, but you have ipykernel 7.1.0 which is incompatible.
spyder-kernels 3.0.5 requires ipython!=8.17.1,<9,>=8.13.0; python_version > "3.8", but you have ipython 9.8.0 which is incompatible.


In [13]:
# ============================================================================
# STEP 2: FASTAPI BACKEND
# File: main.py
# ============================================================================
"""
FastAPI backend for serving hospital staffing predictions
Run with: uvicorn main:app --reload
Access: http://127.0.0.1:8000
"""

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import joblib
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Optional

In [14]:
# LOAD MODEL AND METADATA
# ============================================================================

print("Loading hospital staffing model...")

try:
    model = joblib.load("hospital_staffing_model.pkl")
    feature_columns = joblib.load("feature_columns.pkl")
    metadata = joblib.load("model_metadata.pkl")
    print(" Model loaded successfully")
    print(f"  Model type: {metadata['model_type']}")
    print(f"  Features: {metadata['n_features']}")
    print(f"  Expected MAPE: {metadata['expected_mape']}%")
except Exception as e:
    print(f"✗ Error loading model: {e}")
    raise

Loading hospital staffing model...
 Model loaded successfully
  Model type: Ridge Regression
  Features: 23
  Expected MAPE: 5.35%


In [15]:
# PYDANTIC MODELS FOR API
# ============================================================================

class PredictionInput(BaseModel):
    """Input features for prediction"""
    target_date: str = Field(..., description="Date to predict (YYYY-MM-DD)")
    lag_1: Optional[float] = Field(None, description="Previous day volume")
    lag_7: Optional[float] = Field(None, description="Same day last week")
    rolling_mean_7: Optional[float] = Field(700.0, description="7-day average")
    rolling_std_7: Optional[float] = Field(50.0, description="7-day std dev")
    
    class Config:
        json_schema_extra = {
            "example": {
                "target_date": "2024-06-15",
                "lag_1": 750.0,
                "lag_7": 720.0,
                "rolling_mean_7": 730.0,
                "rolling_std_7": 45.0
            }
        }

class PredictionOutput(BaseModel):
    """Prediction output"""
    predicted_volume: float
    prediction_date: str
    confidence_interval_lower: float
    confidence_interval_upper: float
    mape_expected: float
    staffing_recommendation: str

class ModelInfo(BaseModel):
    """Model metadata"""
    model_type: str
    alpha: float
    n_features: int
    training_samples: int
    training_date: str
    expected_mape: float

# ============================================================================
# CREATE FASTAPI APP
# ============================================================================

app = FastAPI(
    title="Hospital Staffing Prediction API",
    description="ML-powered 7-day advance patient volume forecasting",
    version="1.0.0"
)

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ============================================================================
# API ENDPOINTS
# ============================================================================

@app.get("/")
def home():
    """Welcome endpoint"""
    return {
        "message": "Hospital Staffing Prediction API",
        "version": "1.0.0",
        "model": "Ridge Regression (α=1)",
        "mape": "5.35%",
        "endpoints": {
            "/predict": "POST - Make prediction",
            "/model-info": "GET - Model metadata",
            "/health": "GET - Health check"
        }
    }

@app.get("/health")
def health_check():
    """Health check endpoint"""
    return {
        "status": "healthy",
        "model_loaded": model is not None,
        "timestamp": datetime.now().isoformat()
    }

@app.get("/model-info", response_model=ModelInfo)
def get_model_info():
    """Get model metadata"""
    return ModelInfo(**metadata)

@app.post("/predict", response_model=PredictionOutput)
def predict_volume(input_data: PredictionInput):
    """
    Predict patient volume for a given date
    
    Returns predicted volume with confidence interval and staffing recommendation
    """
    try:
        # Parse target date
        target_date = datetime.strptime(input_data.target_date, '%Y-%m-%d')
        
        # Create feature dataframe
        features_dict = {}
        
        # Temporal features
        features_dict['DayOfWeek'] = target_date.weekday()
        features_dict['DayOfMonth'] = target_date.day
        features_dict['WeekOfYear'] = target_date.isocalendar()[1]
        features_dict['Month'] = target_date.month
        features_dict['IsWeekend'] = 1 if target_date.weekday() >= 5 else 0
        
        # Day-of-week encoding
        for day in range(7):
            features_dict[f'Day_{day}'] = 1 if target_date.weekday() == day else 0
        
        # Lag features (use provided or defaults)
        features_dict['Lag_1'] = input_data.lag_1 or 720.0
        features_dict['Lag_2'] = input_data.lag_1 or 715.0  # Approximate
        features_dict['Lag_3'] = input_data.lag_1 or 710.0  # Approximate
        features_dict['Lag_7'] = input_data.lag_7 or 720.0
        features_dict['Lag_14'] = input_data.lag_7 or 725.0  # Approximate
        
        # Rolling statistics (use provided or defaults)
        features_dict['Rolling_Mean_3'] = input_data.rolling_mean_7 or 720.0
        features_dict['Rolling_Mean_7'] = input_data.rolling_mean_7 or 720.0
        features_dict['Rolling_Mean_14'] = input_data.rolling_mean_7 or 720.0
        features_dict['Rolling_Std_3'] = input_data.rolling_std_7 or 50.0
        features_dict['Rolling_Std_7'] = input_data.rolling_std_7 or 50.0
        features_dict['Rolling_Std_14'] = input_data.rolling_std_7 or 50.0
        
        # Create DataFrame with correct column order
        features_df = pd.DataFrame([features_dict])
        features_df = features_df[feature_columns]  # Ensure correct order
        
        # Make prediction
        prediction = model.predict(features_df)[0]
        
        # Calculate confidence interval (±1.96 * RMSE from notebook: 48.31)
        rmse = 48.31
        ci_lower = prediction - (1.96 * rmse)
        ci_upper = prediction + (1.96 * rmse)
        
        # Staffing recommendation
        if prediction < 650:
            staff_rec = "Low volume expected - Reduce staff to baseline"
        elif prediction < 750:
            staff_rec = "Normal volume expected - Standard staffing"
        elif prediction < 850:
            staff_rec = "High volume expected - Add 1-2 extra staff"
        else:
            staff_rec = "Very high volume expected - Add 3+ extra staff"
        
        return PredictionOutput(
            predicted_volume=round(prediction, 1),
            prediction_date=input_data.target_date,
            confidence_interval_lower=round(ci_lower, 1),
            confidence_interval_upper=round(ci_upper, 1),
            mape_expected=5.35,
            staffing_recommendation=staff_rec
        )
        
    except ValueError as e:
        raise HTTPException(status_code=400, detail=f"Invalid date format: {e}")
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {e}")

@app.post("/predict-week")
def predict_week(start_date: str):
    """
    Predict patient volumes for 7 days starting from start_date
    
    Returns list of predictions for the next week
    """
    try:
        predictions = []
        current_date = datetime.strptime(start_date, '%Y-%m-%d')
        
        # Default values for first prediction
        lag_1 = 720.0
        lag_7 = 720.0
        
        for day in range(7):
            pred_date = (current_date + timedelta(days=day)).strftime('%Y-%m-%d')
            
            input_data = PredictionInput(
                target_date=pred_date,
                lag_1=lag_1,
                lag_7=lag_7,
                rolling_mean_7=720.0,
                rolling_std_7=50.0
            )
            
            result = predict_volume(input_data)
            predictions.append(result.dict())
            
            # Update lags for next prediction
            lag_1 = result.predicted_volume
        
        return {
            "week_start": start_date,
            "predictions": predictions,
            "total_predicted": sum(p['predicted_volume'] for p in predictions)
        }
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Week prediction error: {e}")

In [16]:
# RUN SERVER
# ============================================================================

if __name__ == "__main__":
    import uvicorn
    print("="*80)
    print("Starting Hospital Staffing Prediction API")
    print("="*80)
    print("URL: http://127.0.0.1:8000")
    print("Docs: http://127.0.0.1:8000/docs")
    print("="*80)
    uvicorn.run(app, host="127.0.0.1", port=8000)

Starting Hospital Staffing Prediction API
URL: http://127.0.0.1:8000
Docs: http://127.0.0.1:8000/docs


RuntimeError: asyncio.run() cannot be called from a running event loop